# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Question (Lane 1):** which February signals go with a page's Google visibility dropping in March,
and can a learned model rank declining pages better than my Week 4 baseline rule?

**Methods (three, from simple to strong):**

| Model | How it works, in plain words | Why it fits |
|---|---|---|
| **Logistic Regression** | gives each clue a weight, adds them up, turns the total into a chance of declining | simple and readable: the skill's starting point |
| **Random Forest** | many decision trees built separately on different parts of the data; they vote | the skill's next step ("Logistic Regression, then Random Forest"); stable |
| **Gradient Boosting** | many small trees built one after another; each fixes the previous trees' mistakes | often the strongest; can catch combinations and bent patterns |

All three output a **chance of declining** for every page. I rank pages by that chance and score the top of
the list with **precision@K**, the same metric as my baseline. Lane 1 asks *which signals matter*, so I also
use **permutation importance**: shuffle one clue and see how much worse the model gets.

**What I compare against:** the Week 4 baseline rule (feedly articles first, by February impressions),
random picking, and the base rate, all on the same pages and labels.

**How I pick the "best" model (decided before looking):** the **simplest** model whose precision@100 **and**
precision@500 are both within **2 points** of the best model. Simplest first: Logistic Regression → Random
Forest → Gradient Boosting. A complex model is chosen only if it is clearly better.

**Features (8 columns, all known on 1 March):**

| Feature | Made from | Why |
|---|---|---|
| `is_feedly` | `content_type` | strongest clue in Week 4 (feedly: 53.3% vs 24.8%) |
| `ctr_vs_peers` | Feb clicks, impressions, position | clicks compared with pages at the same rank (CTR-fix idea) |
| `feb_avg_position` | daily position, Feb only | rank in Google (0 treated as missing) |
| `position_is_zero` | `feb_avg_position` | marks the "position 0" data quirk (1,251 pages) |
| `log_feb_impressions` | Feb impressions | visibility; log because a few pages are huge |
| `content_age_days` | `content_created_date` | page age on 1 March |
| `log_search_volume` | `search_volume` | keyword demand; log because it is very skewed |
| `volume_is_missing` | `search_volume` | missing volume declined at 42.9% in Week 4 |

**Left out:** `word_count` (76.5% of pages were updated after 1 March, so it may describe the future), all
March data, and the label.

The code connects to the warehouse and builds the same Feb → Mar frame as Weeks 3 and 4.

In [1]:
# Setup: connect DuckDB to the warehouse (token comes from the Colab Secrets panel, never typed here)
%pip -q install -U "duckdb>=1.1"
import os, getpass
import duckdb
import pandas as pd

# CREATE SECRET needs DuckDB 1.1+. An older copy may already be loaded in this Colab session.
_major, _minor = (int(x) for x in duckdb.__version__.split(".")[:2])
if (_major, _minor) < (1, 1):
    raise RuntimeError(f"DuckDB {duckdb.__version__} is loaded (too old). "
                       "Do Runtime -> Restart session, then Run all again.")
print("DuckDB version:", duckdb.__version__)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
# Only the two months I need (a list of paths; hf:// doesn't support {a,b} globs)
FACT_FEB_MAR = (f"read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
                f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'])")

BEFORE_START, DECISION, AFTER_END = "2026-02-01", "2026-03-01", "2026-03-31"
print('Connected to the warehouse.')

DuckDB version: 2.0.1
Connected to the warehouse.


In [2]:
FEATURES = ["content_age_days", "word_count", "search_volume", "feb_avg_position", "feb_impressions"]

# Build one row per page: Feb features + March outcome
frame = con.sql(f"""
    WITH ok_clients AS (
        SELECT client_hash_id FROM {DIM_CLIENTS}
        WHERE gsc_data_start <= DATE '{BEFORE_START}'
    ),
    per_page AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date <  DATE '{DECISION}') AS feb_impressions,
               SUM(f.gsc_clicks)      FILTER (WHERE f.report_date <  DATE '{DECISION}') AS feb_clicks,
               SUM(f.gsc_avg_position * f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}')
                 / NULLIF(SUM(f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}'), 0)
                                                                                         AS feb_avg_position,
               COUNT(*) FILTER (WHERE f.report_date >= DATE '{DECISION}')                AS mar_days,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date >= DATE '{DECISION}')  AS mar_impressions
        FROM {FACT_FEB_MAR} f
        JOIN ok_clients USING (client_hash_id)
        WHERE f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT p.client_hash_id, p.content_hash_id,
           date_diff('day', d.content_created_date, DATE '{DECISION}') AS content_age_days,
           d.word_count, d.search_volume, d.content_type,
           p.feb_avg_position, p.feb_impressions, p.feb_clicks, p.mar_days, p.mar_impressions
    FROM per_page p
    JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE d.content_created_date < DATE '{BEFORE_START}'
      AND p.feb_impressions >= 100
""").df()

pages_before = len(frame)
frame = frame[frame["mar_days"] > 0].copy()          # pages with no usable March rows: label unknown
dropped_no_march = pages_before - len(frame)

# Label: March impressions per day more than 20% below February's
feb_per_day = frame["feb_impressions"] / 28
mar_per_day = frame["mar_impressions"] / 31
frame["is_declining"] = (mar_per_day < 0.8 * feb_per_day).astype(int)

print(f"Pages in the frame: {len(frame):,} from {frame['client_hash_id'].nunique()} clients")
print(f"Dropped (no usable March data): {dropped_no_march:,}")
print(f"Base rate: {frame['is_declining'].mean() * 100:.1f}% declining\n")
print("Feature summary (no rows shown):")
print(frame[FEATURES].astype(float).describe().T[["count", "mean", "50%", "min", "max"]].round(1).to_string())
print("\nMissing per feature (%):")
print((frame[FEATURES].isna().mean() * 100).round(1).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages in the frame: 65,756 from 26 clients
Dropped (no usable March data): 3,605
Base rate: 25.1% declining

Feature summary (no rows shown):
                    count    mean     50%    min       max
content_age_days  65756.0   211.3   209.0   29.0     464.0
word_count        40863.0  3000.9  2722.0    0.0   10987.0
search_volume     64410.0   183.0    10.0    0.0  246000.0
feb_avg_position  65756.0    10.4     6.9    0.0     104.7
feb_impressions   65756.0  2488.2   823.0  100.0  203401.0

Missing per feature (%):
content_age_days     0.0
word_count          37.9
search_volume        2.0
feb_avg_position     0.0
feb_impressions      0.0


In [3]:
# Feature engineering (everything except ctr_vs_peers, which is built inside each fold)
import numpy as np

frame["is_feedly"] = (frame["content_type"] == "feedly article").astype(int)
frame["position_is_zero"] = (frame["feb_avg_position"] == 0).astype(int)
frame["feb_position"] = frame["feb_avg_position"].where(frame["feb_avg_position"] > 0)   # 0 -> missing
frame["log_feb_impressions"] = np.log10(frame["feb_impressions"].astype(float))
frame["volume_is_missing"] = frame["search_volume"].isna().astype(int)
frame["log_search_volume"] = np.log10(frame["search_volume"].astype(float) + 1)
frame["feb_ctr"] = frame["feb_clicks"] / frame["feb_impressions"] * 100
frame["position_group"] = pd.cut(frame["feb_avg_position"], bins=[0, 1, 3.5, 10.5, 20.5, 50.5, float("inf")],
                                 right=False).astype(str)

MODEL_FEATURES = ["is_feedly", "ctr_vs_peers", "feb_position", "position_is_zero",
                  "log_feb_impressions", "content_age_days", "log_search_volume", "volume_is_missing"]
y = frame["is_declining"].to_numpy()
BASE = y.mean() * 100
print(f"{len(frame):,} pages | base rate {BASE:.1f}% | feedly pages {frame['is_feedly'].sum():,}")
print("Model features:", MODEL_FEATURES)

65,756 pages | base rate 25.1% | feedly pages 704
Model features: ['is_feedly', 'ctr_vs_peers', 'feb_position', 'position_is_zero', 'log_feb_impressions', 'content_age_days', 'log_search_volume', 'volume_is_missing']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Design: 5-fold grouped by client (`GroupKFold`).**

The 26 clients are split into 5 groups. Each round, the models learn from 4 groups and are tested on the
5th. After 5 rounds, **every page has a prediction from models that never saw its client.**

**Why grouped by client:** pages from the same client share habits (same site, same writers). If a client
were in both training and testing, a model could memorize the client instead of learning signals that
travel. One client holds 45% of the feedly pages, so this matters here.

**Why this is honest for my question:**
- **Time order is respected inside each page:** every feature comes from February, the label from March.
- **Peer CTR is built from training clients only,** in every round, so test clients don't leak into it.
- **Same pages, same labels, same metric** as the baseline, so the comparison is fair.

**Limits of this split (I say them now):**
- It tests **new clients**, not **new months**. All rounds use the same Feb → Mar window. Week 6 repeats the
  check on another month pair.
- **The baseline has a small head start:** I picked the feedly clue in Week 4 by looking at all clients,
  including the ones that are test clients here. The models never see test clients. So the baseline's score
  is a little optimistic.
- With only 26 clients, the 5 groups have very different sizes, so results per round will jump around.

The code shows what is in each test round.

**My result (write after Run all):** Very uneven. Rounds 1 and 2 each test only one big client (17,529 and 14,773 pages) and have **no feedly pages**. Rounds 3 to 5 test 4, 11, and 9 clients (about 11,100 pages each) and hold all 704 feedly pages (109, 325, 270). Base rates go from 20.9% to 30.9%. So rounds 1 and 2 cannot test the feedly clue at all, and per-round numbers will jump around.

In [4]:
# The split: 5 rounds, each test round holds whole clients
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(frame, y, groups=frame["client_hash_id"]))
rows = []
for k, (tr, te) in enumerate(folds, 1):
    test = frame.iloc[te]
    assert not set(frame.iloc[tr]["client_hash_id"]) & set(test["client_hash_id"]), "client in both sides"
    rows.append({"round": k, "test_clients": test["client_hash_id"].nunique(), "test_pages": len(test),
                 "base_rate_%": round(test["is_declining"].mean() * 100, 1),
                 "feedly_pages": int(test["is_feedly"].sum())})
print("Every client is tested exactly once, never in its own training data.\n")
print(pd.DataFrame(rows).to_string(index=False))

Every client is tested exactly once, never in its own training data.

 round  test_clients  test_pages  base_rate_%  feedly_pages
     1             1       17529         25.9             0
     2             1       14773         30.9             0
     3             4       11177         20.9           109
     4            11       11137         23.0           325
     5             9       11140         22.7           270


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**What the table shows** (all on the same 65,756 pages and labels):
- **precision@K:** of the top K pages in each ranking, how many really declined. K = 10, 50, 100, 500.
- **AUC:** how well the whole ranking separates declining from not declining (0.5 = random, 1.0 = perfect).
- **Average precision:** a summary of the whole ranking, weighted toward the top (base rate = random).
- Rows: base rate, random picking (20 shuffles), the **baseline rule**, and the three models.

The model rankings combine all 5 test rounds (each page scored by models that never saw its client).
A second table shows precision@50 inside each round, so I can see if a win is steady or carried by one round.

Then the code applies the selection rule I wrote in Section 1.

**My result (write after Run all):** Yes, but only by a little. Logistic Regression got **54.0% at @100 and 51.0% at @500**, against **51.0% and 49.6%** for the baseline (random picking: about 25%). That is +3 and +1.4 points, small enough that luck could explain it. The selection rule picked **Logistic Regression** (the simplest, and also the best at both). Random Forest had the best AUC (0.567) but was worse at the top of the list; Gradient Boosting was the weakest at the top (35.0% at @100). The win is **not steady**: Logistic Regression beat the baseline in 4 of 5 rounds but lost badly in round 1 (20% vs 44%), one client with no feedly pages. Its top 500 are **all feedly pages**, so it mostly re-orders the baseline's pages instead of finding new ones. My honest read: about as good as the baseline at the top, and a bit better over the whole list (AUC 0.545 vs 0.486).

In [5]:
# Train the 3 models in each round; collect out-of-round predictions for every page
import time, sklearn
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
print("scikit-learn", sklearn.__version__, "| seed", SEED)

def make_models():
    return {
        "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                             LogisticRegression(max_iter=2000)),
        "Random Forest": make_pipeline(SimpleImputer(strategy="median"),
                                       RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                                                              n_jobs=-1, random_state=SEED)),
        "Gradient Boosting": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300,
                                                            random_state=SEED),
    }
MODEL_NAMES = list(make_models())

def add_peer_ctr(train_part, test_part):
    # Peer CTR per position group = all clicks / all impressions of TRAINING pages in that group
    g = train_part.groupby("position_group")
    peer = (g["feb_clicks"].sum() / g["feb_impressions"].sum() * 100)
    out = []
    for part in (train_part, test_part):
        p = part["position_group"].map(peer)
        out.append((part["feb_ctr"] / p).where(p > 0))
    return out

oof = {m: np.full(len(frame), np.nan) for m in MODEL_NAMES}
frame["ctr_vs_peers"] = np.nan
fitted = []                                   # (round, models, X_test, y_test) for Section 4
for k, (tr, te) in enumerate(folds, 1):
    t0 = time.time()
    trn, tst = frame.iloc[tr].copy(), frame.iloc[te].copy()
    trn["ctr_vs_peers"], tst["ctr_vs_peers"] = add_peer_ctr(trn, tst)
    frame.loc[frame.index[te], "ctr_vs_peers"] = tst["ctr_vs_peers"].to_numpy()
    models = make_models()
    for name, model in models.items():
        model.fit(trn[MODEL_FEATURES], trn["is_declining"])
        oof[name][te] = model.predict_proba(tst[MODEL_FEATURES])[:, 1]
    fitted.append((k, models, tst[MODEL_FEATURES], tst["is_declining"].to_numpy()))
    print(f"round {k} done in {time.time() - t0:.0f}s")

scikit-learn 1.6.1 | seed 42
round 1 done in 29s
round 2 done in 15s
round 3 done in 15s
round 4 done in 14s
round 5 done in 17s


In [6]:
# The comparison table: base rate, random, baseline rule, 3 models (same pages, same labels)
def precision_at(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

KS = (10, 50, 100, 500)
# Baseline from Week 4: feedly first, then by February impressions
baseline_score = frame["is_feedly"].to_numpy() * 1e12 + frame["feb_impressions"].to_numpy()
rankings = {"Baseline rule (feedly first)": baseline_score, **oof}

rng = np.random.default_rng(SEED)
table = [{"ranking": "Base rate", **{f"P@{k}": round(BASE, 1) for k in KS}, "AUC": 0.5, "avg_precision": round(BASE / 100, 3)}]
table.append({"ranking": "Random picking (20 shuffles)",
              **{f"P@{k}": round(np.mean([precision_at(rng.random(len(y)), y, k) for _ in range(20)]) * 100, 1) for k in KS},
              "AUC": 0.5, "avg_precision": round(BASE / 100, 3)})
for name, s in rankings.items():
    table.append({"ranking": name, **{f"P@{k}": round(precision_at(s, y, k) * 100, 1) for k in KS},
                  "AUC": round(roc_auc_score(y, s), 3), "avg_precision": round(average_precision_score(y, s), 3)})
results = pd.DataFrame(table).set_index("ranking")
print(f"Precision@K in %, all {len(y):,} pages (base rate {BASE:.1f}%):\n")
print(results.to_string())

# Steadiness: precision@50 inside each test round
per_round = []
for k, (tr, te) in enumerate(folds, 1):
    row = {"round": k, "pages": len(te), "base_%": round(y[te].mean() * 100, 1)}
    for name, s in rankings.items():
        row[name.split(" (")[0]] = round(precision_at(np.asarray(s)[te], y[te], 50) * 100, 1)
    per_round.append(row)
print("\nPrecision@50 inside each test round (%):")
print(pd.DataFrame(per_round).to_string(index=False))

# Selection rule (set in Section 1): simplest model within 2 points of the best at BOTH @100 and @500
best100 = max(results.loc[m, "P@100"] for m in MODEL_NAMES)
best500 = max(results.loc[m, "P@500"] for m in MODEL_NAMES)
CHOSEN = next(m for m in MODEL_NAMES
              if results.loc[m, "P@100"] >= best100 - 2 and results.loc[m, "P@500"] >= best500 - 2)
b = results.loc["Baseline rule (feedly first)"]
c = results.loc[CHOSEN]
print(f"\nSelection rule picks: {CHOSEN}")
print(f"{CHOSEN} vs baseline: @100 {c['P@100']} vs {b['P@100']} | @500 {c['P@500']} vs {b['P@500']} "
      f"| AUC {c['AUC']} vs {b['AUC']}")
print("Beats the baseline at BOTH @100 and @500:", bool(c["P@100"] > b["P@100"] and c["P@500"] > b["P@500"]))

# Metrics receipt (small JSON, safe to commit)
import json, os
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/model_metrics.json", "w") as f:
    json.dump({"pages": int(len(y)), "base_rate_pct": round(BASE, 1), "split": "GroupKFold(5) by client",
               "seed": SEED, "sklearn": sklearn.__version__, "chosen_model": CHOSEN,
               "results": results.reset_index().to_dict(orient="records")}, f, indent=2)
print("\nWrote work/outputs/model_metrics.json")

Precision@K in %, all 65,756 pages (base rate 25.1%):

                              P@10  P@50  P@100  P@500    AUC  avg_precision
ranking                                                                     
Base rate                     25.1  25.1   25.1   25.1  0.500          0.251
Random picking (20 shuffles)  22.5  24.2   25.4   24.2  0.500          0.251
Baseline rule (feedly first)  70.0  46.0   51.0   49.6  0.486          0.258
Logistic Regression           80.0  50.0   54.0   51.0  0.545          0.281
Random Forest                 70.0  46.0   47.0   42.2  0.567          0.304
Gradient Boosting             50.0  48.0   35.0   30.2  0.554          0.291

Precision@50 inside each test round (%):
 round  pages  base_%  Baseline rule  Logistic Regression  Random Forest  Gradient Boosting
     1  17529    25.9           44.0                 20.0           40.0               68.0
     2  14773    30.9           38.0                 52.0           44.0               50.0
     3  111

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Three ways to read the chosen model (picked by the rule in Section 3):

1. **What does it lean on?** Permutation importance, measured on test rounds only: shuffle one feature and
   see how much the average precision drops. Bigger drop = the model relies on it more. Logistic Regression
   weights are shown too, because they are easy to read (sign = direction).
2. **Where is it most wrong?** By content type and by position group: the real % declining next to the
   model's average predicted %. A big gap = the model misjudges that group.
3. **Three concrete wrong cases each way:** pages it was most sure would decline but didn't, and pages that
   declined but it scored lowest. No IDs, only client numbers.

**Sanity check (from the skill):** if one feature looks suspiciously perfect, suspect leakage.

**My result (write after Run all):** The top 3 are **February position, is_feedly, and CTR vs peers**, and all three match Week 4: higher-ranked pages declined more, feedly pages declined more, and pages with fewer clicks than their peers declined more. Page age had **negative** importance (shuffling it made the model better on new clients), which agrees with the FALSE verdict on age in Week 4. The model is most wrong for the **"position 0" quirk pages** (predicted 26.4% vs real 42.2%) and **rank 51+ pages** (8.5% vs 21.7%): a straight-line model cannot bend back up at the deep end. Its top 500 come from 7 clients, and one client holds **57.8%**, more than the baseline's 45%. The worst misses had extreme CTR-vs-peers values (about 27 to 62 times their peers), which pushed the score to almost 0%; capping that feature is something to test in Week 6 on another month pair. No feature looked suspiciously perfect (AUC 0.545, far from the 0.994 of my Week 3 leak test), so I see no sign of leakage.

In [8]:
# 1. What does the chosen model lean on? Permutation importance on the test rounds
from sklearn.inspection import permutation_importance

imp = []
for k, models, X_test, y_test in fitted:
    r = permutation_importance(models[CHOSEN], X_test, y_test, scoring="average_precision",
                               n_repeats=5, random_state=SEED, n_jobs=-1)
    imp.append(r.importances_mean)
importance = (pd.Series(np.mean(imp, axis=0), index=MODEL_FEATURES)
              .sort_values(ascending=False).rename("drop_in_avg_precision").round(4))
print(f"Permutation importance ({CHOSEN}, average over 5 test rounds):")
print(importance.to_string())

# Logistic Regression weights (standardized, average over rounds): sign = direction
coefs = np.mean([models["Logistic Regression"][-1].coef_[0] for _, models, _, _ in fitted], axis=0)
print("\nLogistic Regression weights (+ = more likely to decline):")
print(pd.Series(coefs, index=MODEL_FEATURES).sort_values(ascending=False).round(3).to_string())

# 2. Where is it most wrong? Real vs predicted % declining by group
frame["pred"] = oof[CHOSEN]
def gap_table(col):
    t = frame.groupby(col).agg(pages=("is_declining", "size"), real_pct=("is_declining", "mean"),
                               predicted_pct=("pred", "mean"))
    t[["real_pct", "predicted_pct"]] = (t[["real_pct", "predicted_pct"]] * 100).round(1)
    t["gap_pts"] = (t["predicted_pct"] - t["real_pct"]).round(1)
    return t[t["pages"] >= 100]
print("\nBy content type (groups with 100+ pages):")
print(gap_table("content_type").to_string())
print("\nBy February position group:")
print(gap_table("position_group").to_string())

# Who fills the chosen model's top 500?
size_order = frame["client_hash_id"].value_counts().index
frame["client"] = frame["client_hash_id"].map({c: f"client {i + 1}" for i, c in enumerate(size_order)})
top500 = frame.iloc[np.argsort(-frame["pred"].to_numpy(), kind="stable")[:500]]
print(f"\nTop 500 of {CHOSEN}: {top500['client'].nunique()} clients, biggest share "
      f"{top500['client'].value_counts().iloc[0] / 500:.1%}, feedly pages {int(top500['is_feedly'].sum())}")

# 3. Three concrete wrong cases each way
show = ["client", "content_type", "feb_impressions", "feb_avg_position", "ctr_vs_peers",
        "content_age_days", "search_volume", "pred", "is_declining"]
fp = frame[frame["is_declining"] == 0].nlargest(3, "pred")[show]
fn = frame[frame["is_declining"] == 1].nsmallest(3, "pred")[show]
for title, part in [("Most sure it would decline, but it did NOT:", fp),
                    ("Declined, but the model scored it lowest:", fn)]:
    p = part.copy()
    p["pred"] = (p["pred"] * 100).round(1).astype(str) + "%"
    p["feb_avg_position"] = p["feb_avg_position"].round(1)
    p["ctr_vs_peers"] = p["ctr_vs_peers"].round(2)
    print("\n" + title)
    print(p.to_string(index=False))

Permutation importance (Logistic Regression, average over 5 test rounds):
feb_position           0.0194
is_feedly              0.0161
ctr_vs_peers           0.0107
log_search_volume      0.0014
volume_is_missing      0.0011
position_is_zero       0.0000
log_feb_impressions   -0.0005
content_age_days      -0.0133

Logistic Regression weights (+ = more likely to decline):
is_feedly              0.104
volume_is_missing      0.047
content_age_days       0.026
position_is_zero       0.017
log_search_volume     -0.031
log_feb_impressions   -0.108
ctr_vs_peers          -0.143
feb_position          -0.267

By content type (groups with 100+ pages):
                 pages  real_pct  predicted_pct  gap_pts
content_type                                            
feedly article     704      53.3           53.5      0.2
keyword article  65047      24.8           23.9     -0.9

By February position group:
                pages  real_pct  predicted_pct  gap_pts
position_group                         

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.